# Score and cost prediction: L2 Logistic + Ridge

Compare **TF-IDF** and **E5 + four features (388 dimensions)** using training
data only. For each prompt, predict a five-class score distribution and a
dollar cost for each of the 11 candidate LLMs.

**Workflow:** load train → three-fold grouped CV → fit selected parameters
on train → report training errors → save models and predictions.

No calibration, validation, or test files are read. CV chooses parameters
within each representation; final training errors describe the fitted
models and do not establish performance on new prompts or select a model family.

Run top to bottom with `RUN_MODE = "smoke"` first, then use `"full"`.
Smoke uses about 512 training prompts with the same CV grids; its results
are execution checks. Full uses all prepared training prompts.

The final fitted models are saved for a separate inference notebook. Only training-row predictions and training-fold CV predictions are computed here.

**Execution layout:** CV and final training each have separate **TF-IDF** and **E5_388** code cells, with visible headings. Shared helper cells only define computations. After setup, each execution cell processes its named representation only. Rerunning CV replaces that representation's CV tables and clears its old fitted model; rerun its final training, then the shared diagnostics and export. The other representation's results are retained.

**Run status:** Saved outputs below are from the completed `full` run using all 21,900 training prompts. Training errors are in-sample; CV metrics use held-out training folds.

## Call this model from the next notebook

After completing a **full** run, copy the `predict_saved_regression` helper defined at the end of this notebook into the next notebook. Then load the saved package as below; no fitting or CV is required. The example path assumes the next notebook's working directory is the repository root; adjust `run_dir` if needed.

**Example for the next notebook (not executed here):**

```python
from pathlib import Path
import joblib
import numpy as np

# Set this to this method's completed full-run directory.
run_dir = Path("2_Router_Training/experiment_outputs/router_training_score_and_cost_regression/full")
artifact = joblib.load(run_dir / "tfidf_model.joblib")
assert artifact["run_mode"] == "full"
X_new = artifact["preprocessing"]["tfidf_vectorizer"].transform(new_prompt_texts).astype(np.float32)
predictions = predict_saved_regression(artifact, X_new)
llm_names = artifact["model_names"]
```

For E5, load `e5_388_model.joblib` and supply 388-dimensional features
produced by the same Step 1 encoding/chunking recipe. Keep the first 384
semantic coordinates; transform the raw length and two count columns
using the **saved** scalers, then append the code indicator. Do not refit
preprocessing on the new prompts or standardize already scaled columns again.

The helper returns `probability` (rows × 11 × 5), `hard_score`,
`expected_score`, and `dollar_cost` (each rows × 11). Rows follow the input
order and LLM columns follow `artifact["model_names"]`.

## 1. Setup and training data

Required packages: `numpy`, `pandas`, `scipy`, `scikit-learn>=1.6`,
`joblib`, `threadpoolctl`, and `ipykernel`. Use the Step 1 environment when
possible so saved preprocessing objects have compatible versions.

These notebooks are in the project root, beside `data/`. Both input paths
are relative to that root: `data/processed/char_tfidf_50k/` for TF-IDF and
`data/processed/multilingual_e5_small_chunk480/` for E5.

The code automatically finds this root from the kernel's working directory
or its parents. This also works if an existing kernel still starts inside
`2_Router_Training/`. If necessary, set `DATA_PREPARATION_DIR` to the folder
containing `data/`, or set the `ERDOS_DATA_DIR` environment variable.
Outputs remain under `2_Router_Training/experiment_outputs/`; the opening
model-loading examples use that same location.

Inputs are the three `train_*.pkl` tables and `X_train.npz` in
`data/processed/char_tfidf_50k/`, plus `X_train_e5_full_features.npy`
and the two scalers in `data/processed/multilingual_e5_small_chunk480/`.
The TF-IDF preprocessing object supplies feature and LLM column order.
No E5 encoding is rerun.

TF-IDF uses only its text columns. E5 uses 384 semantic coordinates plus
standardized prompt length, choice-marker count, numeric-token count,
and a code indicator. The appended 388-dimensional vector is used as saved.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import json
import os

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from scipy.sparse import load_npz
from sklearn.base import clone
from sklearn.model_selection import GroupKFold

import inspect
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression, Ridge
from threadpoolctl import threadpool_limits

In [2]:
# Run mode and optional directory overrides.
RUN_MODE = "full"  # Use "smoke" for a small end-to-end check.
DATA_PREPARATION_DIR = None  # Folder containing data/; None uses the environment or auto-discovery.
OUTPUT_BASE_OVERRIDE = None  # Optional method-specific output base; the mode subfolder is appended.
WORKFLOW = "router_training_score_and_cost_regression"

# Reproducible training-only CV and smoke sampling.
SEED = 42  # Random seed for grouped folds and the smoke subset.
CV_FOLDS = 3  # Keep each question_group within one fold.
SMOKE_SAMPLE_SIZE = 512  # Base sample size; extra rows cover observed score values.

# Fixed score classes; numerical errors still use the original scores.
score_support = np.array([0.0, 0.25, 0.5, 0.75, 1.0])
NUM_CLASSES = len(score_support)
assert RUN_MODE in {"smoke", "full"}

# Inverse L2 strength for scores: larger C means weaker regularization.
# Select the lowest pooled OOF NLL; exact ties favor smaller C.
C_VALUES = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0]

# L2 strength for costs: larger alpha means stronger regularization.
# Select the lowest pooled OOF cost WAPE; exact ties favor larger alpha.
ALPHA_VALUES = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0]

# Fixed solver and runtime settings; these are not selected by CV.
FIT_N_JOBS = 2  # Number of independent LLM classifiers fitted in parallel.
THREADS_PER_FIT = 1  # CPU threads allowed within each classifier fit.
LOGISTIC_SOLVER = "lbfgs"  # Optimizer for the L2 score classifiers.
LOGISTIC_TOL = 1e-3  # Convergence tolerance for each classifier.
LOGISTIC_ITER_LIMITS = (300, 1000, 2000)  # Per-attempt limits; retries continue from previous weights.
RIDGE_SOLVER = "lsqr"  # Iterative solver for the cost coefficients.
RIDGE_TOL = 1e-3  # Convergence tolerance for the Ridge solver.
RIDGE_MAX_ITER = 1000  # Maximum iterations; reaching this limit triggers a warning.

# Compatibility with scikit-learn's old and new L2 parameter conventions.
L2_OPTIONS = (
    {"penalty": "l2"}
    if inspect.signature(LogisticRegression).parameters["l1_ratio"].default is None
    else {"l1_ratio": 0.0}
)


# Locate the prepared data without changing the search priority.
def find_data_dir():
    requested = DATA_PREPARATION_DIR or os.environ.get("ERDOS_DATA_DIR")
    roots = [Path(requested).expanduser()] if requested else [
        candidate for parent in (Path.cwd(), *Path.cwd().parents)
        for candidate in (parent, parent / "1_data_preparation")
    ]
    required = ("char_tfidf_50k/preprocessing.joblib", "multilingual_e5_small_chunk480/X_train_e5_full_features.npy",
                "multilingual_e5_small_chunk480/length_scaler.joblib", "multilingual_e5_small_chunk480/count_scaler.joblib")
    for root in roots:
        if all((root / "data/processed" / name).is_file() for name in required):
            return root.resolve()
    raise FileNotFoundError("Set DATA_PREPARATION_DIR to the folder containing data/.")

# Resolve data paths and the mode-specific output directory.
preparation_dir = find_data_dir()
processed_dir = preparation_dir / "data" / "processed"
table_dir = processed_dir / "char_tfidf_50k"
e5_dir = processed_dir / "multilingual_e5_small_chunk480"
project_dir = preparation_dir.parent if preparation_dir.name == "1_data_preparation" else preparation_dir
output_base = (Path(OUTPUT_BASE_OVERRIDE).expanduser() if OUTPUT_BASE_OVERRIDE else
               project_dir / "2_Router_Training/experiment_outputs" / WORKFLOW)
OUTPUT_DIR = (output_base / RUN_MODE).resolve()
run_started = datetime.now(timezone.utc).isoformat()
print("Mode:", RUN_MODE, "\nPrepared data:", preparation_dir, "\nOutputs:", OUTPUT_DIR)

# Initialize the tables, fitted models, and predictions for this run.
prediction_bundles, fitted_models, cv_tables, selected_parameters = {}, {}, {}, {}
training_diagnostics_current = False

Mode: full 
Prepared data: C:\Users\johnh\MachineLearning\project_erdos 
Outputs: C:\Users\johnh\MachineLearning\project_erdos\2_Router_Training\experiment_outputs\router_training_score_and_cost_regression\full


In [3]:
preprocessing = joblib.load(table_dir / "preprocessing.joblib")
tfidf_vectorizer = preprocessing["tfidf_vectorizer"]
model_names = list(preprocessing["model_names"])
length_scaler = joblib.load(e5_dir / "length_scaler.joblib")
count_scaler = joblib.load(e5_dir / "count_scaler.joblib")
train_questions, train_scores, train_costs = [
    pd.read_pickle(table_dir / f"train_{name}.pkl")
    for name in ("questions", "scores", "costs")
]
X_train = {
    "tfidf": load_npz(table_dir / "X_train.npz")[:, :len(tfidf_vectorizer.vocabulary_)].astype(np.float32).tocsr(),
    "e5_388": np.load(e5_dir / "X_train_e5_full_features.npy", allow_pickle=False).astype(np.float32),
}
assert len(model_names) == 11
assert train_questions.index.is_unique and train_questions["sample_id"].is_unique
assert train_questions.index.equals(train_scores.index) and train_scores.index.equals(train_costs.index)
assert train_scores.columns.tolist() == train_costs.columns.tolist() == model_names
raw_scores = train_scores.to_numpy(dtype=float)
assert np.isfinite(raw_scores).all() and ((raw_scores >= 0) & (raw_scores <= 1)).all()
raw_costs = train_costs.to_numpy(dtype=float)
assert np.isfinite(raw_costs).all() and (raw_costs > 0).all()
assert X_train["e5_388"].shape == (len(train_questions), 388)
assert X_train["tfidf"].shape[0] == len(train_questions)
assert np.isfinite(X_train["e5_388"]).all() and np.isfinite(X_train["tfidf"].data).all()

# Coverage describes the original training scores before smoke subsampling.
anchor_counts = [(raw_scores == value).sum() for value in score_support]
coverage = pd.DataFrame({"Recorded score": [*score_support, "Other"],
                        "Count": [*anchor_counts, raw_scores.size - sum(anchor_counts)]})
coverage["Share"] = coverage["Count"] / raw_scores.size
display(coverage)
if RUN_MODE == "smoke":
    positions = np.random.default_rng(SEED).choice(len(train_questions), min(SMOKE_SAMPLE_SIZE, len(train_questions)), replace=False)
    examples = [np.argwhere(raw_scores == value)[0, 0] for value in np.unique(raw_scores)]
    positions = np.unique(np.r_[positions, examples])
    train_questions, train_scores, train_costs = [
        frame.iloc[positions].copy() for frame in (train_questions, train_scores, train_costs)
    ]
    X_train = {name: values[positions] for name, values in X_train.items()}
train_score_values = train_scores.to_numpy(dtype=float)
train_cost_values = train_costs.to_numpy(dtype=float)
print("Training prompts:", len(train_questions), "| Feature shapes:", {k: v.shape for k, v in X_train.items()})

# Shared score labels and grouped folds for both representations.
train_class_ids = np.abs(train_score_values[..., None] - score_support).argmin(axis=-1)
groups = train_questions["question_group"].to_numpy()
assert pd.notna(groups).all()
folds = list(GroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED).split(train_questions, groups=groups))
fold_ids = np.zeros(len(train_questions), dtype=int)
for fold_number, (fit_rows, heldout_rows) in enumerate(folds, 1):
    fold_ids[heldout_rows] = fold_number

,Recorded score,Count,Share
0,0.0,93268,0.387165
1,0.25,15064,0.062532
2,0.5,8075,0.033520
3,0.75,27679,0.114898
4,1.0,96473,0.400469
5,Other,341,0.001416


Training prompts: 21900 | Feature shapes: {'tfidf': (21900, 50000), 'e5_388': (21900, 388)}


## 2. Targets, CV, and error definitions

Let $N$ be the number of training prompts, $M=11$ the candidate LLMs,
$x_i$ a prompt's feature vector, and $s_{im},c_{im}$ its recorded score and
dollar cost for LLM $m$. The five score anchors are
$(a_0,\ldots,a_4)=(0,0.25,0.5,0.75,1)$.

Map a recorded score to its nearest anchor:
$z_{im}=\arg\min_{j\in\{0,\ldots,4\}}|s_{im}-a_j|$.
Exact midpoint ties choose the lower anchor. The coverage table explains
the five-class choice; rare intermediate values are approximated for
classification. Numerical errors always use the **original scores**.

Given probabilities $p_{imj}$, the two score outputs are
$$\widehat z_{im}=\arg\max_j p_{imj},\qquad
\widehat s^{\rm hard}_{im}=a_{\widehat z_{im}},\qquad
\widehat s^{\rm expected}_{im}=\sum_{j=0}^4 a_jp_{imj}.$$
Hard-score probability ties choose the lower class. Both outputs share
one probability model and therefore the same selected hyperparameter.

Use three shuffled `GroupKFold` folds (seed 42), grouped by
`question_group`. Each training prompt receives one out-of-fold (OOF)
prediction. TF-IDF and the E5 auxiliary scalers are refitted on the
fitting part of each fold; E5's 384 semantic coordinates remain frozen.
Final fits use the prepared full-training features.

All reported summaries give equal weight to the 11 LLMs. For any evaluated
set of $n$ prompts, with $\mathbf1[\cdot]$ denoting the indicator function:

| Metric (per LLM $m$) | Definition | Direction | Function |
|---|---|---|---|
| Accuracy | $\frac1n\sum_i\mathbf1[\widehat z_{im}=z_{im}]$ | Higher | `accuracy()` |
| Recall for class $j$ | Fraction of true class-$j$ prompts predicted as class $j$ | Higher | `recall()` |
| Balanced accuracy | Mean recall over true classes present in the evaluated rows | Higher | `balanced_accuracy()` |
| NLL (negative log-likelihood) | $-\frac1n\sum_i\log\max(p_{im,z_{im}},10^{-12})$ | Lower | `nll()` |
| Zero true-probability rate | $\frac1n\sum_i\mathbf1[p_{im,z_{im}}=0]$, before the NLL floor | Lower | `zero_true_probability_rate()` |
| MSE | $\frac1n\sum_i(\widehat y_{im}-y_{im})^2$ | Lower | `mse()` |
| MAE | $\frac1n\sum_i\lvert\widehat y_{im}-y_{im}\rvert$ | Lower | `mae()` |
| Cost WAPE | $\frac{\sum_i\lvert\widehat c_{im}-c_{im}\rvert}{\sum_i c_{im}}$ | Lower | `wape()` |

Here $y$ denotes an original score or dollar cost. NLL uses the natural
logarithm. Accuracy measures score-class prediction. WAPE is used only
for costs; `0.10` means 10%. With all five classes present, balanced
accuracy equals one fifth of the sum of their recalls.

The additional `CV_Cost_MSE_normalized` column is
$\frac1{NM}\sum_{i,m}[(\widehat c_{im}-c_{im})/\mu_m^{(-f(i))}]^2$,
where $\mu_m^{(-f(i))}$ is the mean from the fitting part of prompt $i$'s fold.
`normalized_cost_sse()` sums each fold's squared normalized errors;
the pooled sum is divided by $NM$ after all folds.

The next cell implements each metric as a small function. `score_metrics()`
and `cost_metrics()` combine them into the per-LLM tables used by both
CV and training evaluation. Summary tables average the 11 LLMs equally.

In [4]:
# Small metric functions and per-LLM table aggregators.
# Two-dimensional inputs have prompts in rows and LLMs in columns.
def accuracy(classes, predicted):
    return (predicted == classes).mean(axis=0)


def recall(classes, predicted, class_id):
    """Recall for one present class of one LLM (one-dimensional inputs)."""
    return (predicted[classes == class_id] == class_id).mean()


def balanced_accuracy(classes, predicted):
    """Give equal weight to the true classes present for each LLM."""
    return [np.mean([recall(classes[:, m], predicted[:, m], j)
                     for j in np.unique(classes[:, m])]) for m in range(classes.shape[1])]


def nll(true_probabilities):
    """Natural-log loss, with the original probability floor."""
    return -np.log(np.maximum(true_probabilities, 1e-12)).mean(axis=0)


def zero_true_probability_rate(true_probabilities):
    return (true_probabilities == 0).mean(axis=0)


def mse(truth, predictions):
    return np.square(predictions - truth).mean(axis=0)


def mae(truth, predictions):
    return np.abs(predictions - truth).mean(axis=0)


def wape(truth, predictions):
    """Cost WAPE; recorded dollar costs have positive column sums."""
    return np.abs(predictions - truth).sum(axis=0) / truth.sum(axis=0)


def score_metrics(truth, classes, probabilities):
    """Combine score metrics into the per-LLM table used by CV and training."""
    assert probabilities.shape == (*truth.shape, NUM_CLASSES)
    assert np.isfinite(probabilities).all() and (probabilities >= 0).all()
    assert np.allclose(probabilities.sum(axis=-1), 1, atol=1e-6)
    predicted = probabilities.argmax(axis=-1)
    true_p = np.take_along_axis(probabilities, classes[..., None], axis=-1)[..., 0]
    result = pd.DataFrame({
        "Accuracy": accuracy(classes, predicted),
        "Balanced_accuracy": balanced_accuracy(classes, predicted),
        "NLL": nll(true_p),
        "Zero_true_probability_rate": zero_true_probability_rate(true_p),
    }, index=pd.Index(model_names, name="Model"))
    for label, values in {"Hard": score_support[predicted], "Expected": probabilities @ score_support}.items():
        result[f"{label}_MSE"] = mse(truth, values)
        result[f"{label}_MAE"] = mae(truth, values)
    return result


def cost_metrics(truth, predictions):
    """Combine dollar-cost metrics into the same per-LLM table layout."""
    assert predictions.shape == truth.shape and np.isfinite(predictions).all() and (predictions >= 0).all()
    return pd.DataFrame({"MSE": mse(truth, predictions), "MAE": mae(truth, predictions),
                         "WAPE": wape(truth, predictions)},
                        index=pd.Index(model_names, name="Model"))


def normalized_cost_sse(truth, predictions, fitting_means):
    """Accumulate Ridge's normalized squared errors before pooling folds."""
    return np.square((predictions - truth) / fitting_means).sum()


### CV feature preparation

The following two cells prepare TF-IDF and E5_388 separately, using the
grouped folds created after loading training data.

In [5]:
# TF-IDF: fit a fresh vectorizer on the fitting rows of each training fold.
texts = train_questions["question_text"].to_numpy()

def iter_tfidf_cv_features():
    for fold_number, (fit_rows, heldout_rows) in enumerate(folds, 1):
        vectorizer = clone(tfidf_vectorizer)
        X_fit = vectorizer.fit_transform(texts[fit_rows]).astype(np.float32).tocsr()
        X_heldout = vectorizer.transform(texts[heldout_rows]).astype(np.float32).tocsr()
        yield fold_number, fit_rows, heldout_rows, X_fit, X_heldout

In [6]:
# E5_388: keep semantic embeddings fixed and refit the auxiliary scalers per fold.
e5 = X_train["e5_388"]
raw_lengths = length_scaler.inverse_transform(e5[:, 384:385])
raw_counts = count_scaler.inverse_transform(e5[:, 385:387])

def iter_e5_cv_features():
    for fold_number, (fit_rows, heldout_rows) in enumerate(folds, 1):
        lengths = clone(length_scaler).fit(raw_lengths[fit_rows])
        counts = clone(count_scaler).fit(raw_counts[fit_rows])
        def transform(rows):
            return np.column_stack([e5[rows, :384], lengths.transform(raw_lengths[rows]),
                                    counts.transform(raw_counts[rows]), e5[rows, 387:388]]).astype(np.float32)
        X_fit, X_heldout = transform(fit_rows), transform(heldout_rows)
        yield fold_number, fit_rows, heldout_rows, X_fit, X_heldout

def iter_cv_features(embedding):
    # Keep the same interface for the CV searches below.
    builders = {"tfidf": iter_tfidf_cv_features, "e5_388": iter_e5_cv_features}
    return builders[embedding]()

## 3. L2 Logistic score classification

Fit one classifier per LLM. For class $j$, let $w_{mj}$ and $b_{mj}$ be
its feature weights and intercept. With $W_m$ collecting the class weights,

$$p_{mj}(x)=\frac{\exp(b_{mj}+w_{mj}^{\top}x)}
{\sum_{\ell=0}^{4}\exp(b_{m\ell}+w_{m\ell}^{\top}x)},\qquad
\mathcal L_m=-\sum_{i\in I}\log p_{m,z_{im}}(x_i)
+\frac{1}{2C}\lVert W_m\rVert_F^2.$$

$I$ is the fitting set; $C>0$ controls inverse L2 strength. Larger $C$
weakens the weight penalty; intercepts are unpenalized. This summed-loss
convention matches the unweighted scikit-learn objective. Classes absent
from fitting rows receive zero probability in the common five-column output.

Search $C\in\{0.01,0.1,1,10,100,1000,10000\}$ separately for each
representation. Select the lowest pooled OOF mean NLL, breaking exact
ties toward smaller $C$. Hard and expected score MSE/MAE are both reported.
Their numerical errors do not choose separate classifiers.

Preserve `lbfgs`, tolerance $10^{-3}$, and 300 initial iterations with
continuation limits 1,000 and 2,000. Stop if convergence is still unresolved.

In [7]:
# Shared score-fitting and result-management helpers.
def fit_score_head(X, labels, C):
    if np.unique(labels).size == 1:
        return {"classifier": None, "constant_class": int(labels[0]), "iterations": 0}
    classifier = LogisticRegression(C=C, solver=LOGISTIC_SOLVER, tol=LOGISTIC_TOL, random_state=SEED, **L2_OPTIONS)
    iterations = 0
    for limit in LOGISTIC_ITER_LIMITS:
        classifier.set_params(max_iter=limit)
        with threadpool_limits(limits=THREADS_PER_FIT), warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            classifier.fit(X, labels)
        iterations += int(classifier.n_iter_.max())
        for warning in caught:
            if not issubclass(warning.category, ConvergenceWarning):
                warnings.warn(str(warning.message), warning.category)
        if not any(issubclass(w.category, ConvergenceWarning) for w in caught):
            return {"classifier": classifier, "constant_class": None, "iterations": iterations}
        classifier.set_params(warm_start=True)
    raise RuntimeError(f"Logistic did not converge at C={C}; resolve before exporting results.")

def fit_score_heads(X, labels, C):
    return joblib.Parallel(n_jobs=FIT_N_JOBS, prefer="processes", max_nbytes=None, pre_dispatch=FIT_N_JOBS)(
        joblib.delayed(fit_score_head)(X, labels[:, m], C) for m in range(len(model_names)))

def predict_score_heads(heads, X):
    probabilities = np.zeros((X.shape[0], len(model_names), NUM_CLASSES))
    for m, head in enumerate(heads):
        classifier = head["classifier"]
        if classifier is None:
            probabilities[:, m, head["constant_class"]] = 1.0
        else:
            probabilities[:, m, classifier.classes_] = classifier.predict_proba(X)
    return probabilities

def store_embedding_table(name, embedding, frame):
    """Replace this representation's rows while retaining the other one's results."""
    previous = cv_tables.get(name, pd.DataFrame(columns=frame.columns))
    other = previous.loc[previous["Embedding"].ne(embedding)]
    combined = pd.concat([other, frame], ignore_index=True) if len(other) else frame.copy()
    order = combined["Embedding"].map({"tfidf": 0, "e5_388": 1})
    cv_tables[name] = combined.iloc[np.argsort(order.to_numpy(), kind="stable")].reset_index(drop=True)

def invalidate_embedding_fit(embedding):
    """After CV, require a fresh fit for this representation before export."""
    global training_diagnostics_current
    training_diagnostics_current = False
    fitted_models.pop(embedding, None)
    prediction_bundles.pop(embedding, None)
    if "final_fit_diagnostics" in cv_tables:
        table = cv_tables["final_fit_diagnostics"]
        cv_tables["final_fit_diagnostics"] = table.loc[table["Embedding"].ne(embedding)].copy()

In [8]:
# Shared implementation; only the separate calls below execute this stage.
def run_logistic_cv(embedding):
    summary_rows = []
    per_llm_tables = []
    fold_rows = []

    # Store one held-out prediction per training prompt and candidate C.
    oof_probabilities = np.zeros(
        (len(C_VALUES), len(train_questions), len(model_names), NUM_CLASSES)
    )

    for fold, fit_rows, heldout_rows, X_fit, X_heldout in iter_cv_features(embedding):
        for position, C in enumerate(C_VALUES):
            started = perf_counter()

            heads = fit_score_heads(X_fit, train_class_ids[fit_rows], C)
            probabilities = predict_score_heads(heads, X_heldout)
            oof_probabilities[position, heldout_rows] = probabilities

            # Fold metrics are diagnostics; final selection uses pooled OOF rows.
            fold_metrics = score_metrics(
                train_score_values[heldout_rows],
                train_class_ids[heldout_rows],
                probabilities,
            )
            fold_means = fold_metrics.mean()

            fold_rows.append({
                "Embedding": embedding,
                "Fold": fold,
                "C": C,
                "Questions": len(heldout_rows),
                "Max_iterations": max(head["iterations"] for head in heads),
                **fold_means.to_dict(),
                "Seconds": perf_counter() - started,
            })

            print(
                f"Logistic CV / {embedding} / fold {fold} / "
                f"C={C:g}: NLL={fold_means['NLL']:.4f}",
                flush=True,
            )

    # Pool all held-out rows before computing per-LLM metrics.
    for position, C in enumerate(C_VALUES):
        cv_metrics = score_metrics(
            train_score_values,
            train_class_ids,
            oof_probabilities[position],
        ).add_prefix("CV_")

        summary_rows.append({
            "Embedding": embedding,
            "C": C,
            **cv_metrics.mean().to_dict(),
        })
        per_llm_tables.append(
            cv_metrics.reset_index().assign(Embedding=embedding, C=C)
        )

    summary = pd.DataFrame(summary_rows)

    # Minimize NLL; exact ties favor the smaller C.
    best = summary.sort_values(
        ["CV_NLL", "C"], kind="stable"
    ).iloc[0]

    selected_parameters.setdefault(embedding, {})["C"] = float(best["C"])
    summary["Selected"] = summary["C"].eq(best["C"])

    del oof_probabilities

    store_embedding_table(
        "logistic_cv_summary", embedding, summary
    )
    store_embedding_table(
        "logistic_cv_per_llm",
        embedding,
        pd.concat(per_llm_tables, ignore_index=True),
    )
    store_embedding_table(
        "logistic_cv_folds", embedding, pd.DataFrame(fold_rows)
    )

    invalidate_embedding_fit(embedding)
    return summary


### Logistic cross-validation — TF-IDF

In [9]:
# Logistic cross-validation: TF-IDF only.
display(run_logistic_cv("tfidf"))

Logistic CV / tfidf / fold 1 / C=0.01: NLL=0.7913


Logistic CV / tfidf / fold 1 / C=0.1: NLL=0.6734


Logistic CV / tfidf / fold 1 / C=1: NLL=0.6452


Logistic CV / tfidf / fold 1 / C=10: NLL=0.7385


Logistic CV / tfidf / fold 1 / C=100: NLL=1.0767


Logistic CV / tfidf / fold 1 / C=1000: NLL=1.4856


... (12 outputs left out)

Logistic CV / tfidf / fold 3 / C=100: NLL=1.0817


Logistic CV / tfidf / fold 3 / C=1000: NLL=1.3980


Logistic CV / tfidf / fold 3 / C=10000: NLL=1.4939


,Embedding,C,CV_Accuracy,CV_Balanced_accuracy,CV_NLL,CV_Zero_true_probability_rate,CV_Hard_MSE,CV_Hard_MAE,CV_Expected_MSE,CV_Expected_MAE,Selected
0,tfidf,0.01,0.662989,0.401475,0.790557,0.000000,0.261273,0.282146,0.166573,0.361716,False
1,tfidf,0.10,0.679232,0.422647,0.672715,0.000000,0.245621,0.266277,0.156574,0.341097,False
2,tfidf,1.00,0.681789,0.448763,0.644166,0.000000,0.246263,0.265913,0.157322,0.326260,True
3,tfidf,10.00,0.658941,0.456454,0.736697,0.000000,0.266358,0.286645,0.180521,0.317769,False
4,tfidf,100.00,0.637522,0.453801,1.078634,0.000000,0.281597,0.303224,0.216908,0.314614,False
5,tfidf,1000.00,0.635214,0.452633,1.473605,0.000004,0.282150,0.304134,0.230982,0.313462,False
6,tfidf,10000.00,0.632931,0.451359,1.593675,0.000004,0.283481,0.305701,0.234125,0.313715,False


### Logistic cross-validation — E5_388

In [10]:
# Logistic cross-validation: E5_388 only.
display(run_logistic_cv("e5_388"))

Logistic CV / e5_388 / fold 1 / C=0.01: NLL=0.7052


Logistic CV / e5_388 / fold 1 / C=0.1: NLL=0.6645


Logistic CV / e5_388 / fold 1 / C=1: NLL=0.6411


Logistic CV / e5_388 / fold 1 / C=10: NLL=0.6419


Logistic CV / e5_388 / fold 1 / C=100: NLL=0.6452


Logistic CV / e5_388 / fold 1 / C=1000: NLL=0.6472


... (12 outputs left out)

Logistic CV / e5_388 / fold 3 / C=100: NLL=0.6476


Logistic CV / e5_388 / fold 3 / C=1000: NLL=0.6490


Logistic CV / e5_388 / fold 3 / C=10000: NLL=0.6495


,Embedding,C,CV_Accuracy,CV_Balanced_accuracy,CV_NLL,CV_Zero_true_probability_rate,CV_Hard_MSE,CV_Hard_MAE,CV_Expected_MSE,CV_Expected_MAE,Selected
0,e5_388,0.01,0.659128,0.404093,0.705808,0.0,0.262023,0.283544,0.166813,0.360067,False
1,e5_388,0.10,0.676816,0.425959,0.666172,0.0,0.246489,0.267342,0.158669,0.344707,False
2,e5_388,1.00,0.683827,0.444845,0.641707,0.0,0.243945,0.263670,0.156222,0.332596,True
3,e5_388,10.00,0.680137,0.449100,0.642956,0.0,0.247934,0.267551,0.158298,0.327763,False
4,e5_388,100.00,0.678302,0.450245,0.646345,0.0,0.249674,0.269343,0.159606,0.327128,False
5,e5_388,1000.00,0.678053,0.450638,0.647389,0.0,0.249930,0.269589,0.159994,0.326466,False
6,e5_388,10000.00,0.679137,0.452180,0.647485,0.0,0.248963,0.268567,0.159939,0.326785,False


## 4. Ridge dollar-cost regression

Cost is a separate numerical target. For each fitting set $I$, compute
the LLM-specific mean $\mu_m=|I|^{-1}\sum_{i\in I}c_{im}$ and normalized
cost $r_{im}=c_{im}/\mu_m$. With weights $\beta_m$ and intercept $\beta_{0m}$,

$$\min_{\beta_{0m},\beta_m}\sum_{i\in I}
(r_{im}-\beta_{0m}-x_i^\top\beta_m)^2+\alpha\lVert\beta_m\rVert_2^2,
\qquad
\widehat c_m(x)=\mu_m\max(\beta_{0m}+x^\top\beta_m,10^{-6}).$$

$\alpha$ is the L2 penalty strength; the intercept is unpenalized. The
floor prevents negative dollar predictions. Normalization and restoration
use fitting-fold means during CV, then full-training means for the final fit.

Search $\alpha\in\{0.01,0.1,1,10,100,1000,10000\}$ independently of
Logistic's $C$. Pool all OOF dollar errors per LLM, compute each LLM's WAPE,
then average the 11 values. Minimize this mean; exact ties favor larger
$\alpha$. MSE and MAE are additional diagnostics. Preserve `lsqr`,
tolerance $10^{-3}$ and at most 1,000 iterations. A multi-output Ridge
object fits separate coefficients for the 11 LLMs.

In [11]:
# Shared implementation; only the separate calls below execute this stage.
def run_ridge_cv(embedding):
    ridge_per_llm, ridge_folds = [], []
    oof = np.zeros((len(ALPHA_VALUES), *train_cost_values.shape))
    normalized_sse = np.zeros(len(ALPHA_VALUES))
    for fold, fit_rows, heldout_rows, X_fit, X_heldout in iter_cv_features(embedding):
        means = train_cost_values[fit_rows].mean(axis=0)
        targets = (train_cost_values[fit_rows] / means).astype(np.float32)
        for position, alpha in enumerate(ALPHA_VALUES):
            regressor = Ridge(alpha=alpha, solver=RIDGE_SOLVER, tol=RIDGE_TOL, max_iter=RIDGE_MAX_ITER).fit(X_fit, targets)
            predictions = np.maximum(regressor.predict(X_heldout), 1e-6) * means
            oof[position, heldout_rows] = predictions
            normalized_sse[position] += normalized_cost_sse(train_cost_values[heldout_rows], predictions, means)
            metrics = cost_metrics(train_cost_values[heldout_rows], predictions)
            iterations = int(np.max(regressor.n_iter_))
            if iterations >= RIDGE_MAX_ITER:
                warnings.warn(f"Ridge iteration limit: {embedding}, fold {fold}, alpha={alpha:g}.")
            ridge_folds.append({"Embedding": embedding, "Fold": fold, "alpha": alpha,
                                "Max_iterations": iterations, **metrics.add_prefix("Cost_").mean().to_dict()})
        print(f"Ridge CV / {embedding} / fold {fold}: {len(ALPHA_VALUES)} alpha values complete", flush=True)
    rows = []
    for position, alpha in enumerate(ALPHA_VALUES):
        cv_metrics = cost_metrics(train_cost_values, oof[position]).add_prefix("CV_Cost_")
        rows.append({"Embedding": embedding, "alpha": alpha, **cv_metrics.mean().to_dict(),
                     "CV_Cost_MSE_normalized": normalized_sse[position] / train_cost_values.size})
        ridge_per_llm.append(cv_metrics.reset_index().assign(Embedding=embedding, alpha=alpha))
    summary = pd.DataFrame(rows)
    best = summary.sort_values(["CV_Cost_WAPE", "alpha"], ascending=[True, False], kind="stable").iloc[0]
    selected_parameters.setdefault(embedding, {})["alpha"] = float(best["alpha"])
    summary["Selected"] = summary["alpha"].eq(best["alpha"])
    del oof

    store_embedding_table("ridge_cv_summary", embedding, summary)
    store_embedding_table("ridge_cv_per_llm", embedding, pd.concat(ridge_per_llm, ignore_index=True))
    store_embedding_table("ridge_cv_folds", embedding, pd.DataFrame(ridge_folds))
    invalidate_embedding_fit(embedding)
    print(f"Selected parameters / {embedding}: {selected_parameters[embedding]}")
    return summary

### Ridge cross-validation — TF-IDF

In [12]:
# Ridge cross-validation: TF-IDF only.
display(run_ridge_cv("tfidf"))

Ridge CV / tfidf / fold 1: 7 alpha values complete


Ridge CV / tfidf / fold 2: 7 alpha values complete


Ridge CV / tfidf / fold 3: 7 alpha values complete


Selected parameters / tfidf: {'C': 1.0, 'alpha': 10.0}


,Embedding,alpha,CV_Cost_MSE,CV_Cost_MAE,CV_Cost_WAPE,CV_Cost_MSE_normalized,Selected
0,tfidf,0.01,5.118729e-07,0.000265,0.286547,0.195960,False
1,tfidf,0.10,3.979107e-07,0.000213,0.220262,0.140587,False
2,tfidf,1.00,3.565354e-07,0.000174,0.177103,0.121314,False
3,tfidf,10.00,3.840135e-07,0.000166,0.171426,0.128012,True
4,tfidf,100.00,4.884783e-07,0.000198,0.211298,0.159566,False
5,tfidf,1000.00,8.347489e-07,0.000341,0.374405,0.286352,False
6,tfidf,10000.00,1.565986e-06,0.000536,0.567014,0.545980,False


### Ridge cross-validation — E5_388

In [13]:
# Ridge cross-validation: E5_388 only.
display(run_ridge_cv("e5_388"))

Ridge CV / e5_388 / fold 1: 7 alpha values complete


Ridge CV / e5_388 / fold 2: 7 alpha values complete


Ridge CV / e5_388 / fold 3: 7 alpha values complete


Selected parameters / e5_388: {'C': 1.0, 'alpha': 10.0}


,Embedding,alpha,CV_Cost_MSE,CV_Cost_MAE,CV_Cost_WAPE,CV_Cost_MSE_normalized,Selected
0,e5_388,0.01,3.117063e-07,0.000163,0.148584,0.102912,False
1,e5_388,0.10,3.117534e-07,0.000161,0.147695,0.102915,False
2,e5_388,1.00,3.147571e-07,0.000154,0.142098,0.103156,False
3,e5_388,10.00,3.360272e-07,0.000142,0.132934,0.107092,True
4,e5_388,100.00,3.850866e-07,0.000149,0.141204,0.118036,False
5,e5_388,1000.00,4.444175e-07,0.000162,0.157126,0.132722,False
6,e5_388,10000.00,5.558741e-07,0.000227,0.238360,0.172704,False


## 5. Fit selected parameters on training data

Refit the CV-selected classifiers and cost regressors on all rows in the
current run. Predict those same training rows. These are **in-sample
diagnostics**; CV results above remain separate. Smoke and full modes
select and fit their own parameters.

In [14]:
# Shared implementation; only the separate calls below execute this stage.
def fit_regression_embedding(embedding):
    invalidate_embedding_fit(embedding)
    final_fit_rows = []
    cost_means = train_cost_values.mean(axis=0)
    features = X_train[embedding]
    parameters = selected_parameters[embedding]
    heads = fit_score_heads(features, train_class_ids, parameters["C"])
    regressor = Ridge(alpha=parameters["alpha"], solver=RIDGE_SOLVER, tol=RIDGE_TOL, max_iter=RIDGE_MAX_ITER).fit(
        features, (train_cost_values / cost_means).astype(np.float32))
    if np.max(regressor.n_iter_) >= RIDGE_MAX_ITER:
        warnings.warn(f"Final Ridge reached its iteration limit for {embedding}.")
    prediction_bundles[embedding] = {
        "probability": predict_score_heads(heads, features),
        "cost": np.maximum(regressor.predict(features), 1e-6) * cost_means,
    }
    fitted_models[embedding] = {"score_heads": heads, "cost_model": regressor,
                                "cost_means": cost_means, **parameters}
    final_fit_rows.extend({"Embedding": embedding, "Model": name, "Score_iterations": head["iterations"]}
                          for name, head in zip(model_names, heads))
    print(f"Final fit and training predictions complete: {embedding}", flush=True)
    store_embedding_table("final_fit_diagnostics", embedding, pd.DataFrame(final_fit_rows))

### Logistic + Ridge final training — TF-IDF

In [15]:
# Logistic + Ridge final training: TF-IDF only.
fit_regression_embedding("tfidf")

Final fit and training predictions complete: tfidf


### Logistic + Ridge final training — E5_388

In [16]:
# Logistic + Ridge final training: E5_388 only.
fit_regression_embedding("e5_388")

Final fit and training predictions complete: e5_388


## Training diagnostics and baseline

The training-frequency baseline ignores the prompt. For LLM $m$, use
class frequencies $q_{mj}=N^{-1}\sum_i\mathbf1[z_{im}=j]$ and mean cost
$\overline c_m=N^{-1}\sum_i c_{im}$ for every row. Hard and expected
baseline scores follow the same formulas as the fitted models.

Report both per-LLM results and an equal-LLM summary. Training-fit errors
reveal fit quality, including overfitting; they do not measure routing
savings or held-out performance. CV already chose hyperparameters, so
its selected results are also not an independent final test.

In [17]:
training_diagnostics_current = False
assert set(prediction_bundles) == set(X_train), "Run both final-training cells before the shared diagnostics."
frequencies = np.stack([(train_class_ids == j).mean(axis=0) for j in range(NUM_CLASSES)], axis=-1)
baseline = {
    "probability": np.broadcast_to(frequencies, (*train_class_ids.shape, NUM_CLASSES)),
    "cost": np.broadcast_to(train_cost_values.mean(axis=0), train_cost_values.shape),
}
score_tables, cost_tables = [], []
for embedding, bundle in {"Training frequency": baseline, **prediction_bundles}.items():
    score_tables.append(score_metrics(train_score_values, train_class_ids, bundle["probability"])
                        .reset_index().assign(Embedding=embedding))
    cost_tables.append(cost_metrics(train_cost_values, bundle["cost"])
                       .add_prefix("Cost_").reset_index().assign(Embedding=embedding))
train_score_metrics = pd.concat(score_tables, ignore_index=True)
train_cost_metrics = pd.concat(cost_tables, ignore_index=True)
train_score_summary = train_score_metrics.groupby("Embedding", sort=False).mean(numeric_only=True)
train_cost_summary = train_cost_metrics.groupby("Embedding", sort=False).mean(numeric_only=True)
print("In-sample training metrics (equal-LLM averages)")
display(train_score_summary)
display(train_cost_summary)
training_diagnostics_current = True

In-sample training metrics (equal-LLM averages)


,Accuracy,Balanced_accuracy,NLL,Zero_true_probability_rate,Hard_MSE,Hard_MAE,Expected_MSE,Expected_MAE
Embedding,,,,,,,,
Training frequency,0.530992,0.200000,1.187953,0.0,0.305213,0.347055,0.178984,0.382029
tfidf,0.753059,0.508332,0.548164,0.0,0.181008,0.198763,0.124415,0.289289
e5_388,0.693279,0.452918,0.626696,0.0,0.235404,0.254837,0.151657,0.326976


,Cost_MSE,Cost_MAE,Cost_WAPE
Embedding,,,
Training frequency,1.825855e-06,0.000588,0.617193
tfidf,3.415170e-07,0.000158,0.162633
e5_388,3.231031e-07,0.000142,0.132338


## Save this run

Each notebook owns a separate folder under
`2_Router_Training/experiment_outputs/<notebook_name>/`.
`smoke/` and `full/` are separate; rerunning a mode overwrites its filenames.
Existing combined-notebook results stay in their original directory.

| Files | Contents |
|---|---|
| `*_cv_*.csv` | Candidate and per-LLM CV metrics; selected candidates are marked |
| `train_rows.csv` | Sample IDs, prompt groups, and CV fold, in prediction order |
| `train_score_*.csv`, `train_cost_*.csv` | Training metrics and summaries, including baseline |
| `<embedding>_train_class_probability.npy` | Shape: training prompts × 11 LLMs × 5 classes |
| `<embedding>_train_hard_score.pkl`, `..._expected_score.pkl`, `..._dollar_cost.pkl` | Labeled prediction tables |
| `<embedding>_model.joblib` | Complete fitted model package, including selected parameters, transforms, and column order |
| `feature_preprocessing.joblib` | Shared copy of the feature transforms |
| `score_anchor_coverage.csv`, `run_manifest.json` | Original training-score coverage and run metadata |

Prediction rows and LLM columns are saved in their original order.
Model artifacts enable a later, separate inference step. That step must
reproduce the same feature preparation; E5 encoding itself remains in
Step 1. This notebook exports training predictions only. Retain the
manifest when sharing results so smoke output is not confused with full.

In [18]:
assert set(fitted_models) == set(prediction_bundles) == set(X_train), "Run both final-training cells before saving."
assert training_diagnostics_current, "Rerun the shared training diagnostics after the latest CV or final fit before saving."
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest = {
    "workflow": WORKFLOW, "mode": RUN_MODE, "status": "writing", "seed": SEED,
    "started_utc": run_started, "training_questions": len(train_questions),
    "data_splits_loaded": ["train"], "model_artifact_format": 1, "inference_deferred_to_next_step": True, "prediction_split": "train", "evaluation": "in_sample",
    "cv": f"{CV_FOLDS}-fold shuffled GroupKFold on question_group; preprocessing refitted per fold",
    "selected_parameters": selected_parameters, "score_support": score_support.tolist(),
    "model_names": model_names, "versions": {"sklearn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__},
}
manifest_path = OUTPUT_DIR / "run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
# Each entry specifies the filename, table, and whether to retain its index.
csv_outputs = [(f"{name}.csv", frame, False) for name, frame in cv_tables.items()]
csv_outputs += [
    ("train_rows.csv", train_questions[["sample_id", "question_group"]].assign(CV_fold=fold_ids), True),
    ("score_anchor_coverage.csv", coverage, False),
    ("train_score_per_llm.csv", train_score_metrics, False),
    ("train_cost_per_llm.csv", train_cost_metrics, False),
    ("train_score_summary.csv", train_score_summary, True),
    ("train_cost_summary.csv", train_cost_summary, True),
]
for filename, frame, include_index in csv_outputs:
    frame.to_csv(OUTPUT_DIR / filename, index=include_index)

# Save each representation's predictions and complete model package.
for embedding, bundle in prediction_bundles.items():
    probabilities = bundle["probability"]
    np.save(OUTPUT_DIR / f"{embedding}_train_class_probability.npy", probabilities, allow_pickle=False)
    outputs = {"hard_score": score_support[probabilities.argmax(axis=-1)],
               "expected_score": probabilities @ score_support, "dollar_cost": bundle["cost"]}
    for name, values in outputs.items():
        pd.DataFrame(values, index=train_questions.index, columns=model_names).to_pickle(
            OUTPUT_DIR / f"{embedding}_train_{name}.pkl")
    feature_preprocessing = ({"tfidf_vectorizer": tfidf_vectorizer} if embedding == "tfidf" else
                             {"length_scaler": length_scaler, "count_scaler": count_scaler,
                              "semantic_dimension": 384,
                              "auxiliary_feature_order": ["length", "choice_count", "numeric_count", "code_indicator"]})
    artifact = {"format_version": 1, "method": 'regression', "embedding": embedding,
                "run_mode": RUN_MODE, "training_questions": len(train_questions),
                "training_split": "train", "model": fitted_models[embedding],
                "model_names": model_names, "score_support": score_support,
                "selected_parameters": selected_parameters[embedding],
                "n_features": X_train[embedding].shape[1], "preprocessing": feature_preprocessing,
                "versions": manifest["versions"], "normalized_cost_floor": 1e-6}
    joblib.dump(artifact, OUTPUT_DIR / f"{embedding}_model.joblib")
# Save shared preprocessing and mark completion after every output is written.
joblib.dump({"tfidf_vectorizer": tfidf_vectorizer, "e5_length_scaler": length_scaler,
             "e5_count_scaler": count_scaler, "e5_dimension": 388}, OUTPUT_DIR / "feature_preprocessing.joblib")
manifest.update(status="complete", completed_utc=datetime.now(timezone.utc).isoformat(),
                files=sorted(path.name for path in OUTPUT_DIR.iterdir() if path.is_file()))
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved", len(manifest["files"]), "files to", OUTPUT_DIR)

Saved 25 files to C:\Users\johnh\MachineLearning\project_erdos\2_Router_Training\experiment_outputs\router_training_score_and_cost_regression\full


## Reuse the saved model in the next step

The package contains all fitted Logistic coefficients/intercepts and class mappings, Ridge coefficients/intercepts, training cost means, and the selected $C$ and $\alpha$. Saving hyperparameter values alone would not preserve the fitted model.

Each representation has one complete package: `tfidf_model.joblib` or
`e5_388_model.joblib`. It includes the score anchors, LLM column order,
feature transforms, and smoke/full provenance. Use the `full/` artifacts
for the next project stage. Load them in an environment with compatible
package versions; neither CV nor fitting needs to be repeated.

The following cell **defines an inference helper only**. No new dataset
is loaded and no calibration or validation predictions are made here.
Copy the helper into the next notebook with `import numpy as np`.

In [19]:
def predict_saved_regression(artifact, features):
    """Next-step inference on already prepared features; this cell makes no predictions."""
    if features.ndim != 2 or features.shape[1] != artifact["n_features"]:
        raise ValueError("Feature width must match the saved representation.")
    support = np.asarray(artifact["score_support"])
    probabilities = np.zeros((features.shape[0], len(artifact["model_names"]), len(support)))
    costs = np.empty(probabilities.shape[:2])
    if features.shape[0]:
        model = artifact["model"]
        for m, head in enumerate(model["score_heads"]):
            classifier = head["classifier"]
            if classifier is None:
                probabilities[:, m, head["constant_class"]] = 1.0
            else:
                probabilities[:, m, classifier.classes_] = classifier.predict_proba(features)
        costs[:] = np.maximum(model["cost_model"].predict(features), artifact["normalized_cost_floor"]) * model["cost_means"]
    return {"probability": probabilities, "hard_score": support[probabilities.argmax(axis=-1)],
            "expected_score": probabilities @ support, "dollar_cost": costs}